# HarFeast OpenEnv — Multi-Turn GRPO Training

Train `unsloth/Qwen3-4B` to interact with a management consulting environment using tool calls.

**How it works:**
1. The model issues tool calls (JSON actions) to explore data, filter, compute, and submit answers
2. The environment executes each action and returns real observations
3. GRPO generates K trajectories per task, scores them via rubric, and trains on the advantage-weighted difference

This is **multi-turn agent training** — the model learns *which tools to call and when*, not just how to generate a static answer.

## 1. Setup

In [ ]:
# !pip install -q trl datasets accelerate transformers torch wandb
# !git clone https://github.com/17shasvatj/harfeast_apex_openenv_hackathon.git
# %cd harfeast_apex_openenv_hackathon

import json, os, re, sys, time, random, warnings

os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*torch_dtype.*")

if os.path.isdir("harfeast_apex_openenv_hackathon"):
    os.chdir("harfeast_apex_openenv_hackathon")
sys.path.insert(0, os.getcwd())

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from harfeast_openenv.environment import HarFeastOpenEnv
from harfeast_openenv.rubric import score_answer

# W&B setup
use_wandb = True
try:
    import wandb
    os.environ["WANDB_API_KEY"] = "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"
    wandb.init(project="harfeast-grpo", name="notebook-multiturn")
    use_wandb = True
    print("W&B logging enabled")
except Exception as e:
    print(f"W&B disabled: {e}")

MODEL_NAME = "unsloth/Qwen3-4B"
WORLD_PATH = os.path.abspath("harfeast_world")
print(f"Working dir: {os.getcwd()}")
print(f"World: {WORLD_PATH}")
print(f"CUDA: {torch.cuda.is_available()}")

## 2. Environment & Tasks

The HarFeast environment exposes 8 tools: `files.list`, `files.read`, `spreadsheet.read_range`, `data.filter`, `data.group_by`, `data.add_columns`, `data.compute`, `submit`. The agent interacts via JSON actions.

In [ ]:
env = HarFeastOpenEnv(world_path=WORLD_PATH)

with open(os.path.join(WORLD_PATH, "tasks.json")) as f:
    tasks = json.load(f)

print(f"Tasks: {len(tasks)}")
for t in tasks:
    print(f"  {t['task_id']}: {t['task_name']} ({len(t.get('rubric',[]))} criteria)")

# Quick demo: step through one action
obs = env.reset(task_id="task_13")
print(f"\n--- Reset ---\n{obs.observation[:300]}...")

step1 = env.step({"action": "files.list", "path": "."})
print(f"\n--- files.list ---\n{step1.observation[:300]}")

## 3. System Prompt & Action Parser

In [ ]:
SYSTEM_PROMPT = """\
You are a management consultant analyzing data for HarFeast, a food manufacturing company.

You have these tools (output ONLY one JSON object per turn, no other text):

{"action": "files.list", "path": "."}
{"action": "files.read", "path": "documents/report.txt"}
{"action": "spreadsheet.read_range", "file": "data.csv", "range": "1:10"}
{"action": "data.filter", "dataset": "data.csv", "column": "col", "operator": "eq", "value": "x"}
{"action": "data.group_by", "dataset": "data.csv", "column": "col", "aggregation": "sum", "target_column": "val"}
{"action": "data.add_columns", "dataset": "data.csv", "new_column": "total", "expression": "a + b"}
{"action": "data.compute", "expression": "15000 * 0.12"}
{"action": "submit", "answer": "your final answer with specific numbers"}

Operators for data.filter: eq, neq, gt, lt, gte, lte, contains
Aggregations for data.group_by: sum, mean, median, count, min, max
Range for spreadsheet.read_range: "columns" (headers), "1:10" (rows), "all"

Strategy: explore files -> read relevant CSVs -> filter/compute -> submit with numbers.
Output ONLY a JSON object. No explanation text."""


def extract_json_action(text):
    """Parse JSON action from model output, handling thinking tags."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    text = re.sub(r"```json\s*", "", text)
    text = re.sub(r"```\s*$", "", text).strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and "action" in obj: return obj
    except json.JSONDecodeError: pass
    for pat in [r'\{[^{}]*"action"\s*:\s*"[^"]+?"[^{}]*\}', r"\{[^{}]+\}"]:
        m = re.search(pat, text)
        if m:
            try:
                obj = json.loads(m.group())
                if isinstance(obj, dict) and "action" in obj: return obj
            except json.JSONDecodeError: continue
    return None

print("System prompt and parser ready")

## 4. Multi-Turn Rollout Function

This is the core agent loop: model generates an action → environment executes it → observation fed back → repeat until `submit` or max turns.

In [ ]:
# Qwen3 generates <think>...</think> before the actual response.
# With limited tokens, thinking consumes the entire budget and no JSON appears.
# Fix: inject the closing think tag so the model skips straight to content.
THINK_SKIP = "<think>\n</think>\n"

def batched_rollout(model, tokenizer, world_path, task_id, K=4, max_turns=10,
                    temperature=0.8, max_new_tokens=256, force_submit=True):
    """
    Run K trajectories in PARALLEL using batched model.generate().
    Each trajectory gets its own environment instance.
    Returns: (list_of_messages, list_of_rewards, list_of_turns)
    """
    envs = [HarFeastOpenEnv(world_path=world_path) for _ in range(K)]
    all_messages, all_rewards, all_done, all_turns = [], [0.0]*K, [False]*K, [0]*K

    for i in range(K):
        obs = envs[i].reset(task_id=task_id)
        all_messages.append([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": obs.observation},
        ])

    orig_pad_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    for turn in range(max_turns):
        active = [i for i in range(K) if not all_done[i]]
        if not active: break

        # Build prompts with thinking disabled
        prompts = []
        for i in active:
            p = tokenizer.apply_chat_template(
                all_messages[i], tokenize=False, add_generation_prompt=True
            )
            p += THINK_SKIP  # skip Qwen3 thinking phase
            prompts.append(p)

        inputs = tokenizer(
            prompts, return_tensors="pt", truncation=True, max_length=4096, padding=True,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
                do_sample=True, top_p=0.9, pad_token_id=tokenizer.eos_token_id,
            )

        input_len = inputs.input_ids.shape[1]
        for batch_idx, i in enumerate(active):
            new_text = tokenizer.decode(outputs[batch_idx][input_len:], skip_special_tokens=True).strip()
            action_dict = extract_json_action(new_text)
            if action_dict is None:
                all_messages[i].append({"role": "assistant", "content": new_text})
                all_messages[i].append({"role": "user", "content": "Invalid. Output a single JSON object with an 'action' key."})
                all_turns[i] = turn + 1
                continue
            step_result = envs[i].step(action_dict)
            all_messages[i].append({"role": "assistant", "content": json.dumps(action_dict)})
            all_messages[i].append({"role": "user", "content": step_result.observation})
            all_turns[i] = turn + 1
            if step_result.done:
                all_rewards[i] = step_result.reward / 100.0
                all_done[i] = True

    if force_submit:
        for i in range(K):
            if not all_done[i]:
                last_content = ""
                for msg in reversed(all_messages[i]):
                    if msg["role"] == "user" and msg["content"] != "Invalid action. Output a single JSON object with an 'action' key.":
                        last_content = msg["content"][:500]
                        break
                submit_action = {"action": "submit", "answer": last_content}
                step_result = envs[i].step(submit_action)
                all_messages[i].append({"role": "assistant", "content": json.dumps(submit_action)})
                all_messages[i].append({"role": "user", "content": step_result.observation})
                all_rewards[i] = step_result.reward / 100.0
                all_done[i] = True

    tokenizer.padding_side = orig_pad_side
    return all_messages, all_rewards, all_turns


def rollout(model, tokenizer, env, task_id, max_turns=10, temperature=0.8):
    """Single rollout (convenience wrapper)."""
    msgs, rewards, turns = batched_rollout(
        model, tokenizer, env.world_path, task_id, K=1, max_turns=max_turns, temperature=temperature,
    )
    return msgs[0], rewards[0], turns[0]

print(f"Batched rollout ready (thinking disabled, 512 max tokens)")
props = torch.cuda.get_device_properties(0)
total_gb = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / 1e9
print(f"GPU mem: {torch.cuda.memory_allocated()/1e9:.1f}GB / {total_gb:.1f}GB ({props.name})")

## 5. Load Model

In [ ]:
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True,
)
print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters())/1e9:.1f}B")

## 6. Demo: Watch the Agent Interact

In [ ]:
# Run one trajectory on task_13 to see the agent in action
model.eval()
demo_msgs, demo_reward, demo_turns = rollout(
    model, tokenizer, env, "task_13", max_turns=10, temperature=0.3
)
print(f"Reward: {demo_reward:.2f}, Turns: {demo_turns}")
print(f"\n--- Trajectory ---")
for msg in demo_msgs:
    role = msg['role'].upper()
    content = msg['content'][:200]
    if len(msg['content']) > 200: content += '...'
    print(f"\n[{role}]\n{content}")

## 7. Before-Training Evaluation

In [ ]:
def run_eval(model, tokenizer, env, tasks, label="Eval"):
    model.eval()
    seen, unique = set(), []
    for t in tasks:
        if t["task_id"] not in seen:
            seen.add(t["task_id"])
            unique.append(t)
    total_passed, total_criteria = 0, 0
    results = []
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    for i, t in enumerate(unique):
        msgs, reward, turns = rollout(
            model, tokenizer, env, t["task_id"], max_turns=12, temperature=0.3,
        )
        rubric = t.get("rubric", [])
        passed, total = 0, len(rubric)
        score = 0.0
        submitted = env._submitted_answer is not None
        if submitted:
            score, res = score_answer(env._submitted_answer, rubric)
            passed = sum(1 for _, p in res if p)
        total_passed += passed
        total_criteria += total
        results.append({"task_id": t["task_id"], "name": t["task_name"],
                        "passed": passed, "total": total, "score": score,
                        "turns": turns, "submitted": submitted})
        sub = "S" if submitted else "X"
        print(f"  [{i+1}/{len(unique)}] {t['task_id']}  {passed}/{total}  {score:.1f}%  ({turns}t, {sub})")
    overall = (total_passed / total_criteria * 100) if total_criteria > 0 else 0
    n_sub = sum(1 for r in results if r["submitted"])
    print(f"\n  OVERALL: {total_passed}/{total_criteria}  {overall:.1f}%")
    print(f"  Submitted: {n_sub}/{len(unique)}")
    print(f"{'='*60}")
    return round(overall, 1), results

before_score, before_results = run_eval(model, tokenizer, env, tasks, "BEFORE training")
print(f"\nBaseline: {before_score}%")

## 8. Multi-Turn GRPO Training

For each task:
1. Generate K trajectories (model ↔ environment)
2. Score each trajectory with the rubric
3. Compute group-relative advantages (GRPO)
4. Train: increase probability of high-scoring action sequences, decrease low-scoring ones

In [ ]:
def compute_turn_loss(model, tokenizer, messages, turn_index):
    """Cross-entropy loss for one assistant turn in the conversation."""
    prefix_msgs = messages[:turn_index]
    full_msgs = messages[:turn_index + 1]
    prompt_text = tokenizer.apply_chat_template(prefix_msgs, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(full_msgs, tokenize=False)
    prompt_ids = tokenizer(prompt_text, return_tensors="pt").input_ids
    full_ids = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=4096).input_ids.to(model.device)
    if full_ids.shape[1] <= prompt_ids.shape[1]:
        return None
    labels = full_ids.clone()
    labels[:, :prompt_ids.shape[1]] = -100
    outputs = model(full_ids, labels=labels)
    if outputs.loss is None or torch.isnan(outputs.loss):
        return None
    return outputs.loss


def compute_trajectory_loss(model, tokenizer, messages, advantage):
    """GRPO loss = advantage * mean(CE over assistant turns)."""
    total_loss = torch.tensor(0.0, device=model.device)
    n = 0
    for i, msg in enumerate(messages):
        if msg["role"] != "assistant": continue
        loss = compute_turn_loss(model, tokenizer, messages, i)
        if loss is not None:
            total_loss = total_loss + loss
            n += 1
    if n == 0:
        return torch.tensor(0.0, device=model.device, requires_grad=True)
    return advantage * (total_loss / n)

print("GRPO loss functions ready")

In [ ]:
# ── Training loop (batched generation for GPU efficiency) ──
NUM_EPOCHS = 5
NUM_GENERATIONS = 4   # Reduce if OOM (K=8+ needs ~24GB+ VRAM)
MAX_TURNS = 10
LR = 5e-6
TEMPERATURE = 0.8
CHECKPOINT_DIR = "./harfeast_checkpoints_mt"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

global_step = 0
all_epoch_stats = []
print(f"Training: {NUM_EPOCHS} epochs, {NUM_GENERATIONS} gen/task (batched), {MAX_TURNS} max turns")
print(f"Checkpoints: {CHECKPOINT_DIR}\n")

for epoch in range(NUM_EPOCHS):
    print(f"--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    random.shuffle(tasks)
    epoch_rewards = []
    epoch_signal_tasks = 0

    for t_idx, task in enumerate(tasks):
        t_start = time.time()

        model.eval()
        trajectories, rewards, turns_list = batched_rollout(
            model, tokenizer, WORLD_PATH, task["task_id"],
            K=NUM_GENERATIONS, max_turns=MAX_TURNS, temperature=TEMPERATURE,
        )
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        epoch_rewards.extend(rewards)

        mean_r = sum(rewards) / len(rewards)
        var_r = sum((r - mean_r)**2 for r in rewards) / len(rewards)
        std_r = var_r**0.5 + 1e-8
        advantages = [(r - mean_r) / std_r for r in rewards]

        if var_r < 1e-10:
            dt = time.time() - t_start
            print(f"  [{t_idx+1}/{len(tasks)}] {task['task_id']}  r={mean_r:.2f} (no variance) ({dt:.0f}s)")
            continue

        epoch_signal_tasks += 1

        model.train()
        optimizer.zero_grad()
        n_valid = 0
        loss_sum = 0.0
        for traj, adv in zip(trajectories, advantages):
            loss = compute_trajectory_loss(model, tokenizer, traj, adv)
            if loss.requires_grad:
                (loss / len(trajectories)).backward()
                n_valid += 1
                loss_sum += loss.item()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        if n_valid > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            global_step += 1
        optimizer.zero_grad()

        dt = time.time() - t_start
        loss_val = loss_sum / n_valid if n_valid > 0 else 0.0
        print(f"  [{t_idx+1}/{len(tasks)}] {task['task_id']}  r=[{', '.join(f'{r:.2f}' for r in rewards)}]  loss={loss_val:.4f}  ({dt:.0f}s)")

        if use_wandb:
            wandb.log({"train/loss": loss_val, "train/mean_reward": mean_r,
                        "train/max_reward": max(rewards), "train/reward_var": var_r,
                        f"task/{task['task_id']}": mean_r, "train/step": global_step})

    mean_r = sum(epoch_rewards) / max(len(epoch_rewards), 1)
    nonzero = sum(1 for r in epoch_rewards if r > 0)
    all_epoch_stats.append({"epoch": epoch+1, "mean_reward": mean_r, "signal_tasks": epoch_signal_tasks})
    print(f"  Epoch {epoch+1}: mean_reward={mean_r:.3f}, signal={epoch_signal_tasks}/{len(tasks)}, nonzero={nonzero}/{len(epoch_rewards)}")

    if use_wandb:
        wandb.log({"epoch/mean_reward": mean_r, "epoch/signal_tasks": epoch_signal_tasks,
                    "epoch/nonzero_pct": nonzero / max(len(epoch_rewards), 1), "epoch/epoch": epoch+1})

    # Save checkpoint after each epoch
    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"epoch_{epoch+1}")
    os.makedirs(ckpt_path, exist_ok=True)
    model.save_pretrained(ckpt_path)
    tokenizer.save_pretrained(ckpt_path)
    with open(os.path.join(ckpt_path, "train_state.json"), "w") as f:
        json.dump({"epoch": epoch+1, "global_step": global_step, "mean_reward": mean_r,
                    "epoch_stats": all_epoch_stats}, f, indent=2)
    print(f"  Checkpoint saved: {ckpt_path}\n")
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
# Save final model
model.save_pretrained(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)
print(f"Training complete. {global_step} gradient steps. Final model: {CHECKPOINT_DIR}")

## 9. After-Training Evaluation

In [ ]:
after_score, after_results = run_eval(model, tokenizer, env, tasks, "AFTER training")
print(f"\nAfter-training: {after_score}%")

## 10. Before / After Comparison

In [ ]:
delta = after_score - before_score
print(f"{'='*60}")
print(f"  Before: {before_score:.1f}%")
print(f"  After:  {after_score:.1f}%")
print(f"  Delta:  {'+' if delta >= 0 else ''}{delta:.1f}%")
print(f"{'='*60}")

print(f"\n{'Task':<10} {'Name':<30} {'Before':>8} {'After':>8} {'Delta':>8}")
print("-" * 70)
for rb, ra in zip(before_results, after_results):
    d = ra['score'] - rb['score']
    sign = '+' if d >= 0 else ''
    print(f"{rb['task_id']:<10} {rb['name'][:28]:<30} {rb['score']:>6.1f}%  {ra['score']:>6.1f}%  {sign}{d:>6.1f}%")

## 11. Save Model

In [ ]:
OUTPUT_DIR = "./harfeast_checkpoints_mt"
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

results = {
    "model": MODEL_NAME,
    "training_type": "multi-turn GRPO",
    "before_score": before_score,
    "after_score": after_score,
    "delta": delta,
    "before_results": before_results,
    "after_results": after_results,
}
with open(os.path.join(OUTPUT_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved to {OUTPUT_DIR}")